# 03 - Sliding Window and Sticker Masks

This notebook demonstrates sticker identification using sliding window analysis and generates sticker-linker segmentations.

## Contents
1. Compute interaction profiles from intermaps
2. Apply sliding window smoothing
3. Identify stickers via energy thresholds
4. Create sticker masks and visualize segmentation
5. Build sticker-linker chain models

In [ ]:
# Add src to path
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

In [ ]:
# Import modules
from src.sequences import VARIANTS, get_variant
from src.intermaps import (
    compute_all_intermaps,
    InterMapConfig,
    load_intermaps,
    compute_interaction_profile,
)
from src.segmentation import (
    sliding_window_mean,
    sliding_window_gaussian,
    compute_smoothed_interaction_profile,
    identify_stickers_by_energy,
    identify_stickers_by_percentile,
    identify_stickers_by_sequence,
    identify_stickers_hybrid,
    create_sticker_mask,
    StickerMask,
    compute_linker_lengths,
    compute_linker_statistics,
    build_sticker_linker_chain,
    partition_interactions,
    compute_partitioned_energies,
)
from src.plotting import (
    plot_interaction_profile,
    plot_profiles_comparison,
    plot_sticker_linker_diagram,
    plot_sticker_linker_comparison,
    set_publication_style,
    COLOR_STICKER,
    COLOR_LINKER,
)

In [ ]:
set_publication_style()

In [ ]:
# Load or compute intermaps (using unnormalized for physical energies)
config = InterMapConfig(
    smooth=True,
    sigma=2.0,
    normalize=False,  # Keep physical energy units
)
intermaps = compute_all_intermaps(VARIANTS, config=config)
print(f"Loaded {len(intermaps)} interaction maps")

## 1. Interaction Profiles

The interaction profile is a 1D representation showing the mean interaction energy for each residue position.

In [ ]:
# Compute interaction profiles for all variants
profiles = {}
for name, imap in intermaps.items():
    profiles[name] = compute_interaction_profile(imap)
    print(f"{name}: min={profiles[name].min():.3f}, max={profiles[name].max():.3f}")

In [ ]:
# Visualize WT profile with tyrosine positions
wt = get_variant('WT')
fig = plot_interaction_profile(
    profiles['WT'],
    sequence=wt,
    title="FUS LCD Wild-Type: Mean Interaction Profile",
    ylabel="Mean Interaction Energy (kT)",
    highlight_tyrosines=True,
    figsize=(14, 4)
)
plt.show()

In [ ]:
# Compare profiles across variants
fig = plot_profiles_comparison(
    {'WT': profiles['WT'], 'AllY_to_S': profiles['AllY_to_S'], 'AllY_to_F': profiles['AllY_to_F']},
    title="Interaction Profile Comparison: WT vs Y→S vs Y→F",
    figsize=(14, 5)
)
plt.show()

## 2. Sliding Window Smoothing

Additional smoothing of the 1D profile helps identify robust sticker regions.

In [ ]:
# Compare smoothing methods
wt_profile = profiles['WT']

fig, ax = plt.subplots(figsize=(14, 5))
x = np.arange(len(wt_profile)) + 1

ax.plot(x, wt_profile, 'k-', alpha=0.3, label='Raw')
ax.plot(x, sliding_window_mean(wt_profile, window_size=5), 'b-', label='Mean (w=5)')
ax.plot(x, sliding_window_gaussian(wt_profile, sigma=2), 'r-', label='Gaussian (σ=2)')

ax.axhline(y=0, color='gray', linestyle='--', linewidth=0.5)
ax.set_xlabel('Residue Position')
ax.set_ylabel('Mean Interaction Energy (kT)')
ax.set_title('Sliding Window Smoothing of Interaction Profile')
ax.legend()
plt.tight_layout()
plt.show()

## 3. Sticker Identification

We can identify stickers using:
1. **Energy threshold**: Residues with interaction energy below threshold
2. **Percentile threshold**: Bottom N% by interaction energy
3. **Sequence-based**: Known sticker residue types (aromatics, cations)
4. **Hybrid**: Combination of energy and sequence criteria

In [ ]:
# Compare thresholds on WT
wt_profile = profiles['WT']
wt_seq = get_variant('WT')

thresholds = [-0.1, -0.15, -0.2, -0.25]

fig, axes = plt.subplots(len(thresholds), 1, figsize=(14, 2*len(thresholds)), sharex=True)

for ax, thresh in zip(axes, thresholds):
    sticker_bool = identify_stickers_by_energy(wt_profile, threshold=thresh)
    n_stickers = np.sum(sticker_bool)
    
    # Draw bars
    for i in range(len(sticker_bool)):
        color = COLOR_STICKER if sticker_bool[i] else COLOR_LINKER
        ax.axvspan(i, i+1, facecolor=color, edgecolor='none')
    
    ax.set_xlim(0, len(sticker_bool))
    ax.set_ylim(0, 1)
    ax.set_yticks([])
    ax.set_ylabel(f'E<{thresh}', rotation=0, ha='right', va='center')
    ax.set_title(f'Threshold = {thresh} kT → {n_stickers} stickers ({n_stickers/len(sticker_bool):.1%})', loc='left')

axes[-1].set_xlabel('Residue Position')
plt.suptitle('Energy Threshold Effect on Sticker Identification', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Hybrid identification (energy + chemistry)
sticker_hybrid = identify_stickers_hybrid(
    wt_profile, wt_seq,
    energy_threshold=-0.15,
    require_chemistry=True
)

sticker_energy_only = identify_stickers_by_energy(wt_profile, threshold=-0.15)
sticker_chemistry_only = identify_stickers_by_sequence(wt_seq)

print("Sticker Identification Comparison (WT):")
print("=" * 50)
print(f"Energy only (E < -0.15):    {np.sum(sticker_energy_only):3d} stickers ({np.mean(sticker_energy_only):.1%})")
print(f"Chemistry only (Y/F/W/R/K): {np.sum(sticker_chemistry_only):3d} stickers ({np.mean(sticker_chemistry_only):.1%})")
print(f"Hybrid (both criteria):     {np.sum(sticker_hybrid):3d} stickers ({np.mean(sticker_hybrid):.1%})")

## 4. Sticker Masks and Segmentation

In [ ]:
# Create sticker masks for all variants using hybrid method
sticker_masks = {}

for name in VARIANTS:
    seq = VARIANTS[name]
    profile = profiles[name]
    
    # Use hybrid identification
    sticker_bool = identify_stickers_hybrid(
        profile, seq,
        energy_threshold=-0.15,
        require_chemistry=True
    )
    sticker_masks[name] = create_sticker_mask(sticker_bool)
    
    print(f"{name:12s}: {sticker_masks[name].n_stickers:3d} stickers ({sticker_masks[name].sticker_fraction:.1%})")

In [ ]:
# Visualize sticker-linker segmentation for WT
fig = plot_sticker_linker_diagram(
    sticker_masks['WT'],
    sequence=wt_seq,
    title='FUS LCD Wild-Type: Sticker-Linker Segmentation',
    figsize=(14, 2)
)
plt.show()

In [ ]:
# Compare segmentation across variants
fig = plot_sticker_linker_comparison(
    sticker_masks,
    figsize=(14, 8)
)
plt.suptitle('Sticker-Linker Segmentation Across FUS LCD Variants', fontsize=14, y=1.02)
plt.show()

In [ ]:
# Linker statistics
print("Linker Statistics:")
print("=" * 60)

for name, mask in sticker_masks.items():
    stats = compute_linker_statistics(mask)
    print(f"\n{name}:")
    print(f"  Number of linker regions: {stats['n_linkers']}")
    print(f"  Mean linker length: {stats['mean_length']:.1f} residues")
    if stats['n_linkers'] > 0:
        print(f"  Std linker length: {stats['std_length']:.1f} residues")
        print(f"  Range: [{stats['min_length']:.0f}, {stats['max_length']:.0f}]")

## 5. Sticker-Linker Chain Model

In [ ]:
# Build chain model for WT
wt_chain = build_sticker_linker_chain(wt_seq, sticker_masks['WT'])

print("WT Sticker-Linker Chain Model:")
print("=" * 60)
print(f"Total segments: {wt_chain.n_segments}")
print(f"Sticker segments: {wt_chain.n_stickers}")
print(f"Linker segments: {wt_chain.n_linkers}")
print(f"\nChain representation:")
print(wt_chain.to_string_representation())

In [ ]:
# Compare chain representations
print("Chain Representations:")
print("=" * 80)

for name in ['WT', 'AllY_to_S', 'AllY_to_F']:
    chain = build_sticker_linker_chain(VARIANTS[name], sticker_masks[name])
    rep = chain.to_string_representation()
    print(f"\n{name}:")
    print(f"  Segments: {chain.n_segments} ({chain.n_stickers} stickers, {chain.n_linkers} linkers)")
    # Truncate if too long
    if len(rep) > 70:
        print(f"  Pattern: {rep[:70]}...")
    else:
        print(f"  Pattern: {rep}")

In [ ]:
# Partition interaction energies by sticker/linker
print("Partitioned Interaction Energies:")
print("=" * 60)

for name in ['WT', 'AllY_to_S']:
    energies = compute_partitioned_energies(
        intermaps[name],
        sticker_masks[name].mask
    )
    print(f"\n{name}:")
    print(f"  Sticker-Sticker: {energies['E_sticker_sticker']:.4f} kT")
    print(f"  Linker-Linker:   {energies['E_linker_linker']:.4f} kT")
    print(f"  Cross:           {energies['E_cross']:.4f} kT")

In [ ]:
# Save sticker masks
mask_dict = {name: mask.mask for name, mask in sticker_masks.items()}
output_path = Path("../data/outputs/sticker_masks.npz")
np.savez_compressed(output_path, **mask_dict)
print(f"Sticker masks saved to {output_path}")

## Summary

In this notebook we:
1. Computed 1D interaction profiles from intermaps
2. Explored sliding window smoothing methods
3. Identified stickers using energy, sequence, and hybrid criteria
4. Created sticker masks and visualized segmentation patterns
5. Built sticker-linker chain models

**Key observations**:
- WT has ~30 sticker residues concentrated at aromatic positions
- AllY_to_S loses most stickers (reduced to ~10%)
- AllY_to_F retains sticker character (aromatics preserved)

**Next**: Notebook 04 - Difference Maps and Profiles